In [7]:
Here is the complete, clean Python code that can be copied and pasted directly into a Google Colab notebook.

This script generates all three required Excel files (`Participant_Census.xlsx`, `Fund_Performance_Costs.xlsx`, and `Shuster_Advisory_Group_Retirement_Analytics_Model.xlsx`) with professional styling, relational data integrity across multiple retirement plan types (`401(k)`, `403(b)`, `457(b)`, and Defined Benefit), fiduciary benchmark KPIs, and automated browser downloads.

```python
import numpy as np
import openpyxl
import pandas as pd
from google.colab import files
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

# Set random seed for reproducible and professional data modeling
np.random.seed(101)

# ==========================================
# 1. GENERATE PARTICIPANT CENSUS DATA
# ==========================================
n_participants = 350
emp_ids = [f"SHUST-EMP-{1000 + i}" for i in range(n_participants)]
divisions = np.random.choice(
    ["Corporate", "Healthcare (403b)", "Public Sector (457b)", "Education"],
    size=n_participants,
    p=[0.4, 0.25, 0.2, 0.15],
)
plan_types = np.where(
    divisions == "Healthcare (403b)",
    "403(b)",
    np.where(
        divisions == "Public Sector (457b)",
        "457(b)",
        np.random.choice(
            ["401(k)", "Defined Benefit (DB)"], size=n_participants, p=[0.7, 0.3]
        ),
    ),
)

ages = np.random.randint(24, 68, size=n_participants)
salaries = np.random.normal(78000, 22000, size=n_participants).clip(
    45000, 165000
).round(2)
deferral_rates = np.where(
    plan_types == "Defined Benefit (DB)",
    0.0,
    np.random.choice(
        [0.0, 0.03, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15],
        size=n_participants,
        p=[0.12, 0.08, 0.22, 0.22, 0.14, 0.12, 0.05, 0.05],
    ),
)

census_df = pd.DataFrame(
    {
        "Employee_ID": emp_ids,
        "Division": divisions,
        "Plan_Type": plan_types,
        "Age": ages,
        "Annual_Salary": salaries,
        "Deferral_Percentage": deferral_rates,
        "Annual_Contribution": (salaries * deferral_rates).round(2),
    }
)

# Export standalone participant census Excel file
census_df.to_excel(
    "Participant_Census.xlsx", sheet_name="Participant Census", index=False
)

# ==========================================
# 2. GENERATE FUND PERFORMANCE & COSTS DATA
# ==========================================
funds = [
    "Vanguard Institutional Index (S&P 500)",
    "BlackRock Russell 2000 Growth",
    "American Funds EuroPacific Growth",
    "Vanguard Total Bond Market Index",
    "PIMCO Income Fund Institutional",
    "Target Retirement 2030",
    "Target Retirement 2040",
    "Target Retirement 2050",
    "Stable Value Fund",
]
asset_classes = [
    "Large Cap Equity",
    "Small Cap Equity",
    "International Equity",
    "Fixed Income",
    "Fixed Income",
    "Target Date",
    "Target Date",
    "Target Date",
    "Stable Value",
]
beg_aum = np.random.uniform(1500000, 8000000, size=len(funds)).round(2)
net_flows = np.random.uniform(-200000, 500000, size=len(funds)).round(2)
ytd_returns = np.random.uniform(-0.03, 0.16, size=len(funds)).round(4)
expense_ratios = np.random.uniform(
    0.0004, 0.0065, size=len(funds)
).round(4)  # 0.04% to 0.65% expense ratios
end_aum = ((beg_aum + net_flows) * (1 + ytd_returns)).round(2)

funds_df = pd.DataFrame(
    {
        "Fund_Name": funds,
        "Asset_Class": asset_classes,
        "Beginning_AUM": beg_aum,
        "Net_Cash_Flow": net_flows,
        "YTD_Return": ytd_returns,
        "Expense_Ratio": expense_ratios,
        "Ending_AUM": end_aum,
    }
)

# Export standalone fund performance Excel file
funds_df.to_excel(
    "Fund_Performance_Costs.xlsx",
    sheet_name="Fund Performance & Costs",
    index=False,
)

# ==========================================
# 3. BUILD EXECUTIVE FINANCIAL MODEL WORKBOOK
# ==========================================
wb = openpyxl.Workbook()

ws_dash = wb.active
ws_dash.title = "Executive Dashboard"
ws_dash.views.sheetView[0].showGridLines = True

ws_census = wb.create_sheet(title="Participant Census")
ws_census.views.sheetView[0].showGridLines = True

ws_inv = wb.create_sheet(title="Fund Performance & Costs")
ws_inv.views.sheetView[0].showGridLines = True

for row in dataframe_to_rows(census_df, index=False, header=True):
    ws_census.append(row)

for row in dataframe_to_rows(funds_df, index=False, header=True):
    ws_inv.append(row)

# Styling Definitions
navy_header_fill = PatternFill(
    start_color="1B365D", end_color="1B365D", fill_type="solid"
)
white_bold_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
title_font = Font(name="Calibri", size=15, bold=True, color="1B365D")
subtitle_font = Font(name="Calibri", size=11, italic=True, color="555555")
section_font = Font(name="Calibri", size=12, bold=True, color="1B365D")
regular_font = Font(name="Calibri", size=11)
bold_font = Font(name="Calibri", size=11, bold=True)

for ws in [ws_census, ws_inv]:
    for col in range(1, ws.max_column + 1):
        cell = ws.cell(row=1, column=col)
        cell.fill = navy_header_fill
        cell.font = white_bold_font
        cell.alignment = Alignment(horizontal="center", vertical="center")

ws_dash["A1"] = (
    "SHUSTER ADVISORY GROUP | Retirement Plan Analytics & Governance"
)
ws_dash["A1"].font = title_font
ws_dash["A2"] = (
    "Client Plan Sponsor Review: Participation, Performance, and Cost"
    " Efficiency Analysis"
)
ws_dash["A2"].font = subtitle_font

ws_dash["A4"] = "Core Plan Metrics & Fiduciary Benchmarks"
ws_dash["A4"].font = section_font

kpi_headers = ["Metric Description", "Plan Value", "Benchmark Target", "Status"]
for i, h in enumerate(kpi_headers, 1):
    cell = ws_dash.cell(row=5, column=i, value=h)
    cell.fill = navy_header_fill
    cell.font = white_bold_font
    cell.alignment = Alignment(horizontal="center", vertical="center")

kpi_rows = [
    [
        "Overall Plan Participation Rate",
        (
            "=COUNTIF('Participant Census'!F2:F351, \">0\") /"
            " COUNTA('Participant Census'!A2:A351)"
        ),
        0.85,
        "On Track",
    ],
    [
        "Average Employee Deferral Rate",
        "=AVERAGE('Participant Census'!F2:F351)",
        0.075,
        "Review Required",
    ],
    [
        "Total Plan Assets Under Management (AUM)",
        "=SUM('Fund Performance & Costs'!G2:G10)",
        25000000,
        "Optimized",
    ],
    [
        "Weighted Average Fund Expense Ratio",
        (
            "=SUMPRODUCT('Fund Performance & Costs'!G2:G10, 'Fund Performance &"
            " Costs'!F2:F10)/SUM('Fund Performance & Costs'!G2:G10)"
        ),
        0.0035,
        "Cost Efficient",
    ],
    [
        "Multi-Plan Coverage (401k, 403b, 457b, DB)",
        "Active Integration across 4 Plan Types",
        "Multi-Plan Support",
        "Compliant",
    ],
]

for r_idx, r_data in enumerate(kpi_rows, 6):
    ws_dash.cell(row=r_idx, column=1, value=r_data[0]).font = regular_font
    c2 = ws_dash.cell(row=r_idx, column=2, value=r_data[1])
    c2.font = bold_font
    c2.alignment = Alignment(horizontal="right")

    c3 = ws_dash.cell(row=r_idx, column=3, value=r_data[2])
    c3.font = regular_font
    c3.alignment = Alignment(horizontal="right")

    c4 = ws_dash.cell(row=r_idx, column=4, value=r_data[3])
    c4.font = regular_font
    c4.alignment = Alignment(horizontal="center")

ws_dash.cell(row=6, column=2).number_format = "0.0%"
ws_dash.cell(row=6, column=3).number_format = "0.0%"
ws_dash.cell(row=7, column=2).number_format = "0.0%"
ws_dash.cell(row=7, column=3).number_format = "0.0%"
ws_dash.cell(row=8, column=2).number_format = "$#,##0.00"
ws_dash.cell(row=8, column=3).number_format = "$#,##0.00"
ws_dash.cell(row=9, column=2).number_format = "0.00%"
ws_dash.cell(row=9, column=3).number_format = "0.00%"

ws_dash["A13"] = "Participation & Contributions by Retirement Plan Type"
ws_dash["A13"].font = section_font

plan_headers = [
    "Plan Type",
    "Participant Count",
    "Avg. Annual Salary",
    "Avg. Deferral Rate",
    "Total Annual Contributions",
]
for i, h in enumerate(plan_headers, 1):
    cell = ws_dash.cell(row=14, column=i, value=h)
    cell.fill = navy_header_fill
    cell.font = white_bold_font
    cell.alignment = Alignment(horizontal="center", vertical="center")

plan_rows = [
    [
        "401(k)",
        '=COUNTIF(\'Participant Census\'!C2:C351, "401(k)")',
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "401(k)", \'Participant'
            " Census\'!E2:E351)"
        ),
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "401(k)", \'Participant'
            " Census\'!F2:F351)"
        ),
        (
            '=SUMIF(\'Participant Census\'!C2:C351, "401(k)", \'Participant'
            " Census\'!G2:G351)"
        ),
    ],
    [
        "403(b)",
        '=COUNTIF(\'Participant Census\'!C2:C351, "403(b)")',
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "403(b)", \'Participant'
            " Census\'!E2:E351)"
        ),
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "403(b)", \'Participant'
            " Census\'!F2:F351)"
        ),
        (
            '=SUMIF(\'Participant Census\'!C2:C351, "403(b)", \'Participant'
            " Census\'!G2:G351)"
        ),
    ],
    [
        "457(b)",
        '=COUNTIF(\'Participant Census\'!C2:C351, "457(b)")',
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "457(b)", \'Participant'
            " Census\'!E2:E351)"
        ),
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "457(b)", \'Participant'
            " Census\'!F2:F351)"
        ),
        (
            '=SUMIF(\'Participant Census\'!C2:C351, "457(b)", \'Participant'
            " Census\'!G2:G351)"
        ),
    ],
    [
        "Defined Benefit (DB)",
        '=COUNTIF(\'Participant Census\'!C2:C351, "Defined Benefit (DB)")',
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "Defined Benefit'
            " (DB)\", 'Participant Census'!E2:E351)"
        ),
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "Defined Benefit'
            " (DB)\", 'Participant Census'!F2:F351)"
        ),
        (
            '=SUMIF(\'Participant Census\'!C2:C351, "Defined Benefit'
            " (DB)\", 'Participant Census'!G2:G351)"
        ),
    ],
]

for row_idx, r_data in enumerate(plan_rows, 15):
    for col_idx, val in enumerate(r_data, 1):
        cell = ws_dash.cell(row=row_idx, column=col_idx, value=val)
        cell.font = regular_font
        if col_idx == 2:
            cell.number_format = "#,##0"
            cell.alignment = Alignment(horizontal="right")
        elif col_idx == 3:
            cell.number_format = "$#,##0"
            cell.alignment = Alignment(horizontal="right")
        elif col_idx == 4:
            cell.number_format = "0.0%"
            cell.alignment = Alignment(horizontal="right")
        elif col_idx == 5:
            cell.number_format = "$#,##0.00"
            cell.alignment = Alignment(horizontal="right")

for ws in wb.worksheets:
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 14)

wb.save("Shuster_Advisory_Group_Retirement_Analytics_Model.xlsx")

# ==========================================
# 4. TRIGGER BROWSER DOWNLOADS
# ==========================================
files.download("Participant_Census.xlsx")
files.download("Fund_Performance_Costs.xlsx")
files.download("Shuster_Advisory_Group_Retirement_Analytics_Model.xlsx")

print("All Excel files successfully generated and downloaded.")

```

SyntaxError: invalid syntax (2905356669.py, line 1)

In [9]:
import numpy as np
import openpyxl
import pandas as pd
from google.colab import files
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

# Set random seed for reproducible and professional data modeling
np.random.seed(101)

# ==========================================
# 1. GENERATE PARTICIPANT CENSUS DATA
# ==========================================
n_participants = 350
emp_ids = [f"SHUST-EMP-{1000 + i}" for i in range(n_participants)]
divisions = np.random.choice(
    ["Corporate", "Healthcare (403b)", "Public Sector (457b)", "Education"],
    size=n_participants,
    p=[0.4, 0.25, 0.2, 0.15],
)
plan_types = np.where(
    divisions == "Healthcare (403b)",
    "403(b)",
    np.where(
        divisions == "Public Sector (457b)",
        "457(b)",
        np.random.choice(
            ["401(k)", "Defined Benefit (DB)"], size=n_participants, p=[0.7, 0.3]
        ),
    ),
)

ages = np.random.randint(24, 68, size=n_participants)
salaries = np.random.normal(78000, 22000, size=n_participants).clip(
    45000, 165000
).round(2)
deferral_rates = np.where(
    plan_types == "Defined Benefit (DB)",
    0.0,
    np.random.choice(
        [0.0, 0.03, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15],
        size=n_participants,
        p=[0.12, 0.08, 0.22, 0.22, 0.14, 0.12, 0.05, 0.05],
    ),
)

census_df = pd.DataFrame(
    {
        "Employee_ID": emp_ids,
        "Division": divisions,
        "Plan_Type": plan_types,
        "Age": ages,
        "Annual_Salary": salaries,
        "Deferral_Percentage": deferral_rates,
        "Annual_Contribution": (salaries * deferral_rates).round(2),
    }
)

# Export standalone participant census Excel file
census_df.to_excel(
    "Participant_Census.xlsx", sheet_name="Participant Census", index=False
)

# ==========================================
# 2. GENERATE FUND PERFORMANCE & COSTS DATA
# ==========================================
funds = [
    "Vanguard Institutional Index (S&P 500)",
    "BlackRock Russell 2000 Growth",
    "American Funds EuroPacific Growth",
    "Vanguard Total Bond Market Index",
    "PIMCO Income Fund Institutional",
    "Target Retirement 2030",
    "Target Retirement 2040",
    "Target Retirement 2050",
    "Stable Value Fund",
]
asset_classes = [
    "Large Cap Equity",
    "Small Cap Equity",
    "International Equity",
    "Fixed Income",
    "Fixed Income",
    "Target Date",
    "Target Date",
    "Target Date",
    "Stable Value",
]
beg_aum = np.random.uniform(1500000, 8000000, size=len(funds)).round(2)
net_flows = np.random.uniform(-200000, 500000, size=len(funds)).round(2)
ytd_returns = np.random.uniform(-0.03, 0.16, size=len(funds)).round(4)
expense_ratios = np.random.uniform(
    0.0004, 0.0065, size=len(funds)
).round(4)
end_aum = ((beg_aum + net_flows) * (1 + ytd_returns)).round(2)

funds_df = pd.DataFrame(
    {
        "Fund_Name": funds,
        "Asset_Class": asset_classes,
        "Beginning_AUM": beg_aum,
        "Net_Cash_Flow": net_flows,
        "YTD_Return": ytd_returns,
        "Expense_Ratio": expense_ratios,
        "Ending_AUM": end_aum,
    }
)

# Export standalone fund performance Excel file
funds_df.to_excel(
    "Fund_Performance_Costs.xlsx",
    sheet_name="Fund Performance & Costs",
    index=False,
)

# ==========================================
# 3. BUILD EXECUTIVE FINANCIAL MODEL WORKBOOK
# ==========================================
wb = openpyxl.Workbook()

ws_dash = wb.active
ws_dash.title = "Executive Dashboard"
ws_dash.views.sheetView[0].showGridLines = True

ws_census = wb.create_sheet(title="Participant Census")
ws_census.views.sheetView[0].showGridLines = True

ws_inv = wb.create_sheet(title="Fund Performance & Costs")
ws_inv.views.sheetView[0].showGridLines = True

for row in dataframe_to_rows(census_df, index=False, header=True):
    ws_census.append(row)

for row in dataframe_to_rows(funds_df, index=False, header=True):
    ws_inv.append(row)

# Styling Definitions
navy_header_fill = PatternFill(
    start_color="1B365D", end_color="1B365D", fill_type="solid"
)
white_bold_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
title_font = Font(name="Calibri", size=15, bold=True, color="1B365D")
subtitle_font = Font(name="Calibri", size=11, italic=True, color="555555")
section_font = Font(name="Calibri", size=12, bold=True, color="1B365D")
regular_font = Font(name="Calibri", size=11)
bold_font = Font(name="Calibri", size=11, bold=True)

for ws in [ws_census, ws_inv]:
    for col in range(1, ws.max_column + 1):
        cell = ws.cell(row=1, column=col)
        cell.fill = navy_header_fill
        cell.font = white_bold_font
        cell.alignment = Alignment(horizontal="center", vertical="center")

ws_dash["A1"] = (
    "SHUSTER ADVISORY GROUP | Retirement Plan Analytics & Governance"
)
ws_dash["A1"].font = title_font
ws_dash["A2"] = (
    "Client Plan Sponsor Review: Participation, Performance, and Cost"
    " Efficiency Analysis"
)
ws_dash["A2"].font = subtitle_font

ws_dash["A4"] = "Core Plan Metrics & Fiduciary Benchmarks"
ws_dash["A4"].font = section_font

kpi_headers = ["Metric Description", "Plan Value", "Benchmark Target", "Status"]
for i, h in enumerate(kpi_headers, 1):
    cell = ws_dash.cell(row=5, column=i, value=h)
    cell.fill = navy_header_fill
    cell.font = white_bold_font
    cell.alignment = Alignment(horizontal="center", vertical="center")

kpi_rows = [
    [
        "Overall Plan Participation Rate",
        (
            "=COUNTIF('Participant Census'!F2:F351, \">0\") /"
            " COUNTA('Participant Census'!A2:A351)"
        ),
        0.85,
        "On Track",
    ],
    [
        "Average Employee Deferral Rate",
        "=AVERAGE('Participant Census'!F2:F351)",
        0.075,
        "Review Required",
    ],
    [
        "Total Plan Assets Under Management (AUM)",
        "=SUM('Fund Performance & Costs'!G2:G10)",
        25000000,
        "Optimized",
    ],
    [
        "Weighted Average Fund Expense Ratio",
        (
            "=SUMPRODUCT('Fund Performance & Costs'!G2:G10, 'Fund Performance &"
            " Costs'!F2:F10)/SUM('Fund Performance & Costs'!G2:G10)"
        ),
        0.0035,
        "Cost Efficient",
    ],
    [
        "Multi-Plan Coverage (401k, 403b, 457b, DB)",
        "Active Integration across 4 Plan Types",
        "Multi-Plan Support",
        "Compliant",
    ],
]

for r_idx, r_data in enumerate(kpi_rows, 6):
    ws_dash.cell(row=r_idx, column=1, value=r_data[0]).font = regular_font
    c2 = ws_dash.cell(row=r_idx, column=2, value=r_data[1])
    c2.font = bold_font
    c2.alignment = Alignment(horizontal="right")

    c3 = ws_dash.cell(row=r_idx, column=3, value=r_data[2])
    c3.font = regular_font
    c3.alignment = Alignment(horizontal="right")

    c4 = ws_dash.cell(row=r_idx, column=4, value=r_data[3])
    c4.font = regular_font
    c4.alignment = Alignment(horizontal="center")

ws_dash.cell(row=6, column=2).number_format = "0.0%"
ws_dash.cell(row=6, column=3).number_format = "0.0%"
ws_dash.cell(row=7, column=2).number_format = "0.0%"
ws_dash.cell(row=7, column=3).number_format = "0.0%"
ws_dash.cell(row=8, column=2).number_format = "$#,##0.00"
ws_dash.cell(row=8, column=3).number_format = "$#,##0.00"
ws_dash.cell(row=9, column=2).number_format = "0.00%"
ws_dash.cell(row=9, column=3).number_format = "0.00%"

ws_dash["A13"] = "Participation & Contributions by Retirement Plan Type"
ws_dash["A13"].font = section_font

plan_headers = [
    "Plan Type",
    "Participant Count",
    "Avg. Annual Salary",
    "Avg. Deferral Rate",
    "Total Annual Contributions",
]
for i, h in enumerate(plan_headers, 1):
    cell = ws_dash.cell(row=14, column=i, value=h)
    cell.fill = navy_header_fill
    cell.font = white_bold_font
    cell.alignment = Alignment(horizontal="center", vertical="center")

plan_rows = [
    [
        "401(k)",
        '=COUNTIF(\'Participant Census\'!C2:C351, "401(k)")',
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "401(k)", \'Participant'
            " Census\'!E2:E351)"
        ),
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "401(k)", \'Participant'
            " Census\'!F2:F351)"
        ),
        (
            '=SUMIF(\'Participant Census\'!C2:C351, "401(k)", \'Participant'
            " Census\'!G2:G351)"
        ),
    ],
    [
        "403(b)",
        '=COUNTIF(\'Participant Census\'!C2:C351, "403(b)")',
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "403(b)", \'Participant'
            " Census\'!E2:E351)"
        ),
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "403(b)", \'Participant'
            " Census\'!F2:F351)"
        ),
        (
            '=SUMIF(\'Participant Census\'!C2:C351, "403(b)", \'Participant'
            " Census\'!G2:G351)"
        ),
    ],
    [
        "457(b)",
        '=COUNTIF(\'Participant Census\'!C2:C351, "457(b)")',
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "457(b)", \'Participant'
            " Census\'!E2:E351)"
        ),
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "457(b)", \'Participant'
            " Census\'!F2:F351)"
        ),
        (
            '=SUMIF(\'Participant Census\'!C2:C351, "457(b)", \'Participant'
            " Census\'!G2:G351)"
        ),
    ],
    [
        "Defined Benefit (DB)",
        '=COUNTIF(\'Participant Census\'!C2:C351, "Defined Benefit (DB)")',
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "Defined Benefit'
            " (DB)\", 'Participant Census'!E2:E351)"
        ),
        (
            '=AVERAGEIF(\'Participant Census\'!C2:C351, "Defined Benefit'
            " (DB)\", 'Participant Census'!F2:F351)"
        ),
        (
            '=SUMIF(\'Participant Census\'!C2:C351, "Defined Benefit'
            " (DB)\", 'Participant Census'!G2:G351)"
        ),
    ],
]

for row_idx, r_data in enumerate(plan_rows, 15):
    for col_idx, val in enumerate(r_data, 1):
        cell = ws_dash.cell(row=row_idx, column=col_idx, value=val)
        cell.font = regular_font
        if col_idx == 2:
            cell.number_format = "#,##0"
            cell.alignment = Alignment(horizontal="right")
        elif col_idx == 3:
            cell.number_format = "$#,##0"
            cell.alignment = Alignment(horizontal="right")
        elif col_idx == 4:
            cell.number_format = "0.0%"
            cell.alignment = Alignment(horizontal="right")
        elif col_idx == 5:
            cell.number_format = "$#,##0.00"
            cell.alignment = Alignment(horizontal="right")

for ws in wb.worksheets:
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 14)

wb.save("Shuster_Advisory_Group_Retirement_Analytics_Model.xlsx")

# ==========================================
# 4. TRIGGER BROWSER DOWNLOADS
# ==========================================
files.download("Participant_Census.xlsx")
files.download("Fund_Performance_Costs.xlsx")
files.download("Shuster_Advisory_Group_Retirement_Analytics_Model.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>